# 品质理论：转移矩阵、纯回收机与组装-回收环

官方文档：https://wiki.factorio.com/Quality/zh

## 1 基础转移矩阵

### 1.1 两个表达式

输出 = 输入 × 转移矩阵：

$$g = f\,M$$

转移矩阵 = 出入比例 × 品质矩阵（$Q$ 取 **Quality** 的首字母）：

$$M = \mathrm{diag}(r)\,Q(q)$$

### 1.1.1 记号与约定

- **品质等级**：叙述一律用 $1$–$5$ 的等级编号——$1$=普通、$2$=优秀、$3$=稀有、$4$=史诗、$5$=传奇（口语“$k$ 星”即等级 $k$）；代码实现里再用 $0$-index（$0..4$）与之对应。
- $i \in \{1,2,3,4,5\}$：品质等级指标，记传奇档 $L := 5$。
- $q$：单次制作的品质概率（最终 roll 概率），由设备插槽上的品质插件加成叠加而来：若有 $4$ 个插槽、每个品质插件提供 $2.5\%$，则 $q = 4\times 2.5\% = 10\%$，即 $q=\text{插槽数}\times\text{单插件倍率}$（同规格插件时；不同规格取各自倍率之和）。
- $r$：**出入比例**——产出组数 / 投入组数（已含全部产能来源，按“一组”计，与配方件数无关，例如 $1$ 铜块 $\to$ $2$ 铜线的配方 $r$ 依然是 $1$）。$r$ 的来源枚举如下：
  1. **设备产能**：部分设备自带产能，如铸造厂、电磁工厂；
  2. **产能插件**：插在设备上的产能插件加成；
  以上两项合并计入组装侧：$r = 1+P$，其中 $P$ 为“设备自带 + 插件”的总产能加成（如 $50\%$ 产能即 $P=0.5$）；
  3. **回收机**：固定保留 $\tfrac14$（湮灭 $\tfrac34$），即 $r = \tfrac14$（游戏定值，不参数化）。
- $e_i$：第 $i$ 个分量为 $1$ 的列向量（$5\times 1$）；$e_L$ 即传奇收集指示向量；$\mathbf 1$ 为全 $1$ 列向量。
- **流体**：不受品质机制约束——**不是**令 $q=0$，而是在本理论中**没有任何表现**（没有任何表达式与流体相关）。
- 即使有一天要考虑流体，这里也不会变化——$1$ 份原材料 $\to$ $1$ 份产品仍然是 $M = \mathrm{diag}(1)$。此处仅作游戏机制的提示。

### 1.1.2 状态向量与转移矩阵

- $f = (f_1,f_2,f_3,f_4,f_5)$：$f_k$ 为等级 $k$ 的物品量。
- 单位是“**组**”——产出 $1$ 件产品所需的若干原料算 $1$ 组（如 $1$ 铜块 $\to$ $2$ 铜线时，$2$ 根铜线为 $1$ 组），因此量与配方件数无关。
- $M$ 的第 $i$ 行第 $j$ 列：输入品质 $i$ 时产出品质 $j$ 的**期望倍数**（$=$ 概率 $\times$ 出入比例）。
- $g = fM$ 即作用一次后各等级的量；多级设备串联即矩阵连乘。
- 循环系统展开为无限长直线链。

### 1.1.3 特例：普通生产、组装机与回收机的基础矩阵

在 $g = fM$ 中取 $M = \mathrm{diag}(r)\,Q(q)$，按 $r$ 的来源特例化：

- **普通生产**（无品质、无产能）：$M = I = \mathrm{diag}(1)$，品质与数量都不变。
- **组装机**：$M = \mathrm{diag}(1+P)\,Q(q)$，$P$ 为设备自带产能（铸造厂、电磁工厂等）与产能插件的合计。
- **回收机**：$M = \mathrm{diag}(\tfrac14)\,Q(q)$，固定湮灭 $\tfrac34$（游戏定值，不参数化）。

三者都可直接左乘状态：$g = fM$。

可作为练习：取具体 $f$ 与 $q$ 手算一遍 $g$，并检验 $Q(q)$ 行和为 $1$（品质滚动本身不改变件数，变化只来自 $r$）。

若各等级配置不同（如低等级挂品质插件、高等级挂产能插件），可逐行取 $(q_i, r_i)$；细节留待模拟代码建模。

### 1.2 品质矩阵 $Q(q)$

$$
Q(q) =
\begin{pmatrix}
1-q & 9q\cdot10^{-1} & 9q\cdot10^{-2} & 9q\cdot10^{-3} & q\cdot10^{-3}\\
0 & 1-q & 9q\cdot10^{-1} & 9q\cdot10^{-2} & q\cdot10^{-2}\\
0 & 0 & 1-q & 9q\cdot10^{-1} & q\cdot10^{-1}\\
0 & 0 & 0 & 1-q & q\\
0 & 0 & 0 & 0 & 1
\end{pmatrix}.
$$

### 1.2.1 滚动机制、分段通式与校验

**滚动机制**（官方）：

- 先以概率 $q$ 首次判定；成功则升一档；
- 再以固定 $10\%$ 的概率继续判定，直到失败；传奇为吸收态；
- **不可能跳档**（连续升级是逐档累乘 $1/10$ 的几何因子，而非品质加成的幂次）。

由此得到分段通式：

$$
Q(q)_{ij} =
\begin{cases}
0, & j < i,\\[4pt]
1-q, & j = i,\\[4pt]
\dfrac{9q}{10}\cdot 10^{-(j-i-1)}, & i < j < L,\\[6pt]
q\cdot 10^{-(L-i-1)}, & j = L > i.
\end{cases}
$$

以等级 $1$（普通）行为例，与 wiki 表格逐项一致：升 1 档 $\tfrac{9}{10}q$，升 2 档 $\tfrac{9}{100}q$，升 3 档 $\tfrac{9}{1000}q$，直升传奇 $\tfrac{1}{1000}q$。

**注意**：$Q(q)$ 是一般转移矩阵 $M$ 的特例——只有品质滚动、出入比例 $r\equiv 1$；加上产能或湮灭后即 $M = \mathrm{diag}(r)\,Q(q)$。

### 1.3 范围：不考虑速度与节能插件

模型只考虑**品质插件**与**产能插件**（分别进入 $q$ 与 $r$）。

- **速度插件**：不考虑。它只会降低 $q$（挤占品质插件的槽位），对 $r$ 没有影响；吞吐不足完全可以用**多铺设备**补足，速度只是工程侧变量。
- **节能插件**：不考虑。电能消耗不需要优化，只要**供电充足**即可，模型不引入能耗约束。

## 2 纯回收机循环

模型背景：把物品反复送入回收机，目标是收集传奇产物（例如“铁板 $\to$ 传奇铁板”的自循环）。

回收机湮灭 $3/4$ 的物品，未湮灭的以品质分布返回，传奇则被移出系统。

### 2.1 设定与湮灭率

- 回收机固定保留 $\tfrac14$（湮灭 $\tfrac34$）：游戏定值、无需参数化；组归一化下与配方件数无关。
- 回收机的品质概率 $q$ 由品质插件决定。
- 进料为行向量 $f$（$1\times 5$，通常 $f_L = 0$；若 $f_L>0$，传奇进料会因下行清零而直接离开系统，即被收集）。

### 2.2 收集矩阵与回收矩阵

系统把**收集**（传奇产品移出系统）与**继续回收**（非传奇部分再过机）分开处理。定义**收集矩阵**（Collect）

$$
C := \mathrm{diag}(0,0,0,0,1),
$$

它只保留传奇（5 星）分量。

$g = fM$ 是产出向量，则 $gC = fMC$ 就是产出中的传奇产品向量；剩余部分用**通过掩码**表示

$$
S := I - C = \mathrm{diag}(1,1,1,1,0).
$$

**回收矩阵**（$R$ 取 **recycle** 的首字母；不预先清行）为

$$
R := \tfrac14\, Q(q).
$$

一次过机的分解：产出为 $fR$，其中被收集走的是 $fRC$，继续留在循环里的是 $fRS$。

（行向量约定下掩码体现为右乘以筛选分量；列向量约定下对应形式为左乘。）

### 2.3 循环展开、闭式与数值截断

对画图与模拟，循环用编译原理的**循环展开**处理：把回收机的逐次通过展开成有限长的链即可，

$$
fR \;\to\; \underbrace{fRC}_{\text{收集}} \;+\; \underbrace{fRS}_{\text{继续}},
\qquad
fRSR \;\to\; \underbrace{fRSRC}_{\text{收集}} \;+\; \underbrace{fRSRS}_{\text{继续}},\quad \cdots
$$

代数式则直接求和。总收集（各等级的传奇产出向量 $\mathbf y$）为

$$
\mathbf y \;=\; fRC + fRSRC + fRSRSRC + \cdots \;=\; fR\bigl(I + SR + (SR)^2 + \cdots\bigr)C
\;=\; fR\,(I - SR)^{-1}\,C,
$$

- 标量产出 $y = \mathbf y\,\mathbf 1 = fR(I-SR)^{-1}C\,\mathbf 1$。
- 非传奇存量为 $s_k = f(SR)^k$（$(SR)^k$ 的传奇分量恒为 $0$）。

因为每过一次机都有 $\tfrac34$ 被湮灭，$(SR)^k$ 的行和以 $(\tfrac14)^k$ 几何衰减：$\|SR\|_\infty \le \|R\|_\infty \le \tfrac14 < 1$，级数收敛，链“不那么长”。

- 数值模拟可以很快截断（逐段累加直到余项小于容差）。
- 矩阵等比数列求和 $(I-SR)^{-1}$ 与截断等价，**实际实现建议用数值截断**。

## 3 组装机 + 回收机循环

模型背景：组装机把材料制成产品，非传奇产品送回收机拆回材料，材料再回组装机。

只把**传奇产品**移出系统（传奇材料继续作为组装机输入，最终仍以传奇产品形式离开）。

### 3.1 组装矩阵 $A = \mathrm{diag}(1+P)\,Q(q)$

$A$ 取 **Assembling**（组装）的首字母。

$P$ 为产能加成（设备自带产能 + 产能插件），$Q(q)$ 为品质矩阵：

$$
A = \mathrm{diag}(1+P)\,Q(q).
$$

即组装机特例的一般形式。

### 3.2 单轮

就一轮（不迭代、不更新进料）：进料 $f$ 组装后，产出 $fA$ 分两路——

$$
fA \;\to\; \underbrace{fAC}_{\text{收集}} \;+\; \underbrace{fAS}_{\text{非传奇}}
\;\to\; \underbrace{fASR}_{\text{回收产出}} .
$$

注意：$fASR$ 不被直接收集——它是材料，收集只作用于产品。第二轮把同样的因子再乘一遍：材料 $fASR$ 组装为 $fASRA$，收集 $fASRAC$，非传奇 $fASRAS$ 回收 $fASRASR$，即每轮都是同样的 $AC$（收集）与 $ASR$（组装-掩码-回收）。

注（与 wiki 的关联）：wiki 用 10 维轮转矩阵 $\begin{pmatrix}0&A\\SR&0\end{pmatrix}$ 记账（前 5 材料、后 5 产品，列向量约定下需转置），把材料、产品拼成一个状态；

其 $T^2$ 的左上角即 $ASR$，与这里的 5 维一轮链等价。

### 3.3 闭式与级数

把每轮的收集直接加起来（逐轮展开即得）：

$$
y \;=\; fAC + fASRAC + fASRASRAC + \cdots
\;=\; fA\bigl(I + SRA + (SRA)^2 + \cdots\bigr)C
\;=\; fA\,(I - SRA)^{-1}\,C .
$$

- $y$ 为各品质等级的传奇产出向量，传奇总量取其 $L$ 分量（或右乘 $\mathbf 1$）。
- 同理：每轮都湮灭掉一大块，级数快速收敛，工程实现直接数值截断（逐轮累加直到余项小于容差）即可。
- 收敛要求 $(1+P)\cdot\tfrac14 < 1$，即产能加成上限约 $300\%$——超过后系统不再收敛。